# 01 · Data Collection & Processing
**Project**: War Shocks & Global Financial Risk Spillover  
**Period**: 2000-01-01 → 2025-12-31  
**Output**: `data/processed/` — all_variables_aligned, log_returns, rolling_volatility, war_dummies

**Equity universe (8 indices)**

| Name | Ticker | Market |
|------|--------|--------|
| SP500 | ^GSPC | US |
| DAX | ^GDAXI | Germany |
| CAC40 | ^FCHI | France |
| FTSE100 | ^FTSE | UK |
| Nikkei | ^N225 | Japan |
| KOSPI | ^KS11 | South Korea |
| HangSeng | ^HSI | Hong Kong |
| SSE | 000001.SS | China (Shanghai) |

All data collection and processing logic lives in `src/data_loader.py`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import (
    load_yahoo_equity, load_yahoo_controls, load_fred,
    merge_and_align, compute_returns, compute_volatility,
    build_war_dummies, save,
)

START = "2000-01-01"
END   = "2025-12-31"

## Step 1 · Load Equity Indices

In [2]:
print("=" * 55)
print("Yahoo Finance — Equity Indices")
print("=" * 55)
equity_frames = load_yahoo_equity(START, END)

Yahoo Finance — Equity Indices
  [OK] SP500      (^GSPC       ): 6538 rows | 2000-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] DAX        (^GDAXI      ): 6601 rows | 2000-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] CAC40      (^FCHI       ): 6644 rows | 2000-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] FTSE100    (^FTSE       ): 6566 rows | 2000-01-04 ~ 2025-12-30 | missing=0.0%
  [OK] Nikkei     (^N225       ): 6369 rows | 2000-01-04 ~ 2025-12-30 | missing=0.0%
  [OK] KOSPI      (^KS11       ): 6403 rows | 2000-01-04 ~ 2025-12-30 | missing=0.0%
  [OK] HangSeng   (^HSI        ): 6405 rows | 2000-01-03 ~ 2025-12-30 | missing=0.0%
  [OK] SSE        (000001.SS   ): 6294 rows | 2000-01-04 ~ 2025-12-30 | missing=0.0%


## Step 2 · Load Control Variables

In [3]:
print("=" * 55)
print("Yahoo Finance — Safe-haven & Control Variables")
print("=" * 55)
control_frames = load_yahoo_controls(START, END)

Yahoo Finance — Safe-haven & Control Variables
  [OK] Gold     (GC=F        ): 6357 rows
  [OK] DXY      (DX-Y.NYB    ): 6567 rows
  [OK] Silver   (SI=F        ): 6359 rows


## Step 3 · Load FRED Macro Variables

In [4]:
print("=" * 55)
print("FRED — Local Excel")
print("=" * 55)
fred_df = load_fred(START, END)

FRED — Local Excel
  [FIX] Brent: replaced 183 zero(s) → NaN
  [FIX] WTI: replaced 263 zero(s) → NaN
  [FIX] US10Y: replaced 281 zero(s) → NaN
  [FIX] HY_OAS: replaced 82 zero(s) → NaN
  [OK] FRED: 6782 rows | 2000-01-04 ~ 2025-12-31
    VIX       : 3.18% missing
    Brent     : 2.70% missing
    WTI       : 3.88% missing
    US10Y     : 4.14% missing
    HY_OAS    : 1.21% missing


## Step 4 · Merge & Align to S&P 500 Trading Days

In [5]:
print("=" * 55)
print("Merging & Aligning")
print("=" * 55)
combined = merge_and_align(equity_frames, control_frames, fred_df)

print(f"\nFinal shape : {combined.shape}")
print(f"Date range  : {combined.index[0].date()} ~ {combined.index[-1].date()}")
combined.head(3)

Merging & Aligning
  Trading days (S&P 500 basis): 6538

Final shape : (6538, 16)
Date range  : 2000-01-03 ~ 2025-12-30


,SP500,DAX,CAC40,FTSE100,Nikkei,KOSPI,HangSeng,SSE,Gold,DXY,Silver,VIX,Brent,WTI,US10Y,HY_OAS
Date,,,,,,,,,,,,,,,,
2000-01-03,1455.219971,6750.759766,5917.370117,NaN,NaN,NaN,17369.630859,NaN,NaN,100.220001,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,1399.420044,6586.950195,5672.020020,6665.899902,19002.859375,1059.040039,17072.820312,1406.370972,NaN,100.410004,NaN,27.01,23.95,25.56,6.49,4.81
2000-01-05,1402.109985,6502.069824,5479.700195,6535.899902,18542.550781,986.309998,15846.719727,1409.682007,NaN,100.379997,NaN,26.41,23.72,24.65,6.62,4.77


## Step 5 · Compute Returns & Rolling Volatility

In [6]:
returns_df = compute_returns(combined)
vol_df     = compute_volatility(returns_df, window=21)

print(f"Returns    shape : {returns_df.shape}")
print(f"Volatility shape : {vol_df.shape}")
print("\nReturns (equity only) — descriptive stats:")
equity_ret_cols = [
    c for c in returns_df.columns
    if c.endswith("_ret")
    and not any(x in c for x in ["Gold", "DXY", "Silver", "Brent", "WTI"])
]
returns_df[equity_ret_cols].describe().round(6)

Returns    shape : (6538, 16)
Volatility shape : (6538, 8)

Returns (equity only) — descriptive stats:


d:\Miniconda\envs\python3_12_D200\Lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


,SP500_ret,DAX_ret,CAC40_ret,FTSE100_ret,Nikkei_ret,KOSPI_ret,HangSeng_ret,SSE_ret
count,6537.000000,6537.000000,6537.000000,6536.000000,6532.000000,6525.000000,6537.000000,6325.000000
mean,0.000238,0.000197,0.000049,0.000061,0.000151,0.000211,0.000061,0.000137
std,0.012216,0.014257,0.013852,0.011306,0.014445,0.014282,0.014760,0.014397
min,-0.127652,-0.130549,-0.130983,-0.115117,-0.132341,-0.161154,-0.146954,-0.127636
25%,-0.004736,-0.006113,-0.006272,-0.004846,-0.006542,-0.005623,-0.006766,-0.005774
50%,0.000641,0.000587,0.000299,0.000323,0.000000,0.000189,0.000000,0.000000
75%,0.005876,0.007067,0.006938,0.005545,0.007470,0.007026,0.007248,0.006440
max,0.109572,0.107975,0.105946,0.093842,0.132346,0.112844,0.134068,0.094010


## Step 6 · Build War Dummy Variables

**Middle East wars** (from `war_events.xlsx`):
- Iraq War (2003)
- Lebanon War 2006 ← high-intensity
- Gaza War 2008 ← high-intensity
- Gaza War 2014
- ISIS Iraq escalation (2014)
- US-Iran crisis (2020)
- Israel-Hamas War 2023 ← high-intensity

**Crisis periods** (hard-coded):
- `gfc_crisis`: 2008-09-01 – 2009-06-30
- `covid_crisis`: 2020-02-01 – 2020-09-30

In [7]:
print("=" * 55)
print("War Event Dummies")
print("=" * 55)
war_dummy = build_war_dummies(combined.index, END)
war_dummy.value_counts().sort_index()

War Event Dummies
  mideast_war    days : 271
  high_intensity days : 14
  gfc_crisis     days : 209
  covid_crisis   days : 168
  any_crisis     days : 377


mideast_war  high_intensity  gfc_crisis  covid_crisis  any_crisis
0            0               0           0             0             5904
                                         1             1              168
                             1           0             1              195
1            0               0           0             0              257
             1               1           0             1               14
Name: count, dtype: int64

## Step 7 · Save to `data/processed/`

In [8]:
print("Saving processed files...")
save(combined,    "all_variables_aligned.xlsx")
save(returns_df,  "log_returns.xlsx")
save(vol_df,      "rolling_volatility.xlsx")
save(war_dummy,   "war_dummies.xlsx")
print("\nAll files saved to data/processed/")

Saving processed files...
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\all_variables_aligned.xlsx  (6538, 16)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\log_returns.xlsx  (6538, 16)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\rolling_volatility.xlsx  (6538, 8)
  Saved → c:\Users\cn_55\Desktop\上课文件\2 - D200\Problem Set\D200_PS\Project\data\processed\war_dummies.xlsx  (6538, 5)

All files saved to data/processed/


---
**Next** → `02_eda.ipynb` for exploratory analysis  